In [1]:
"""
Enhanced Target Labeling for AI Crypto Trading Bot
Creates realistic and timeframe-aware target labels for machine learning.
"""
import pandas as pd
import numpy as np
import os
import sys
import json
import logging
from datetime import datetime, timezone, timedelta
from typing import Dict, List, Optional, Tuple
import warnings
# Add this cell at the very beginning of your notebook
import sys
import os

# Fix Unicode encoding for Windows
if sys.platform == "win32":
    os.environ['PYTHONIOENCODING'] = 'utf-8'
    
# Also update your logging setup to avoid emojis
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('target_labeling.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
# Add src to path for imports
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir) if 'notebooks' in current_dir else current_dir
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('target_labeling.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuration
FEATURES_FOLDER = 'data/features'
LABELED_FOLDER = 'data/labeled'
BASE_SYMBOLS = ['BTC_USDT', 'ETH_USDT', 'SOL_USDT']

# Timeframe-specific settings for realistic targets
TIMEFRAME_SETTINGS = {
    '6h': {
        'profit_target_pct': 2.0,
        'stop_loss_pct': 1.0,
        'lookforward_periods': 4,  # 24 hours
        'grace_periods': 4         # 24 hours grace
    },
    '12h': {
        'profit_target_pct': 4.0,
        'stop_loss_pct': 2.0,
        'lookforward_periods': 4,  # 48 hours
        'grace_periods': 4         # 48 hours grace
    },
    '1d': {
        'profit_target_pct': 8.0,
        'stop_loss_pct': 4.0,
        'lookforward_periods': 5,  # 5 days
        'grace_periods': 3         # 3 days grace
    },
    '3d': {
        'profit_target_pct': 15.0,
        'stop_loss_pct': 8.0,
        'lookforward_periods': 5,  # 15 days
        'grace_periods': 2         # 6 days grace
    }
}

class EnhancedTargetLabeler:
    """Enhanced target labeling with realistic and adaptive targets"""
    
    def __init__(self):
        self.labeling_stats = {}
        # Ensure labeled folder exists
        os.makedirs(LABELED_FOLDER, exist_ok=True)
    
    def load_features(self, symbol: str, timeframe: str) -> Optional[pd.DataFrame]:
        """Load features for a symbol and timeframe"""
        try:
            filename = f"features_{symbol}_{timeframe}.csv"
            filepath = os.path.join(FEATURES_FOLDER, filename)
            
            if not os.path.exists(filepath):
                logger.warning(f"Features file not found: {filepath}")
                return None
            
            df = pd.read_csv(filepath, parse_dates=['timestamp'])
            df = df.sort_values('timestamp').reset_index(drop=True)
            
            logger.info(f"Loaded {len(df)} feature rows for {symbol} {timeframe}")
            return df
            
        except Exception as e:
            logger.error(f"Error loading features for {symbol} {timeframe}: {e}")
            return None
    
    def calculate_dynamic_targets(self, df: pd.DataFrame, timeframe: str) -> Tuple[float, float]:
        """Calculate dynamic profit and stop loss targets based on volatility"""
        try:
            # Get base settings
            settings = TIMEFRAME_SETTINGS.get(timeframe, TIMEFRAME_SETTINGS['1d'])
            base_profit = settings['profit_target_pct']
            base_stop = settings['stop_loss_pct']
            
            # Calculate recent volatility (ATR-based)
            if 'atr_pct' in df.columns:
                recent_volatility = df['atr_pct'].tail(20).mean()
                if pd.notna(recent_volatility) and pd.api.types.is_numeric_dtype(df['atr_pct'].dtype):
                    try:
                        # Ensure we have a real number, not complex
                        if isinstance(recent_volatility, (int, float)) and not isinstance(recent_volatility, complex):
                            volatility_val = float(recent_volatility)
                            if volatility_val > 0:
                                # Adjust targets based on volatility
                                volatility_multiplier = max(0.5, min(2.0, volatility_val / 2.0))
                                
                                dynamic_profit = base_profit * volatility_multiplier
                                dynamic_stop = base_stop * volatility_multiplier
                                
                                return dynamic_profit, dynamic_stop
                    except (ValueError, TypeError, OverflowError):
                        pass
            
            # Fallback to historical volatility
            if 'close' in df.columns and len(df) > 20:
                returns = df['close'].pct_change().tail(20)
                volatility = returns.std() * 100  # Convert to percentage
                
                if pd.notna(volatility) and pd.api.types.is_numeric_dtype(returns.dtype):
                    try:
                        # Ensure we have a real number, not complex
                        if isinstance(volatility, (int, float)) and not isinstance(volatility, complex):
                            volatility_val = float(volatility)
                            if volatility_val > 0:
                                volatility_multiplier = max(0.5, min(2.0, volatility_val / 3.0))
                                
                                dynamic_profit = base_profit * volatility_multiplier
                                dynamic_stop = base_stop * volatility_multiplier
                                
                                return dynamic_profit, dynamic_stop
                    except (ValueError, TypeError, OverflowError):
                        pass
            
            # Fallback to base settings
            return base_profit, base_stop
            
        except Exception as e:
            logger.warning(f"Error calculating dynamic targets: {e}")
            settings = TIMEFRAME_SETTINGS.get(timeframe, TIMEFRAME_SETTINGS['1d'])
            return settings['profit_target_pct'], settings['stop_loss_pct']
    
    def detect_breakout_setup(self, df: pd.DataFrame, i: int, timeframe: str) -> bool:
        """Detect if current position shows breakout setup characteristics"""
        try:
            if i < 20:  # Need some history
                return False
            
            # Get recent data window
            window_data = df.iloc[max(0, i-20):i+1]
            
            # Initialize counters
            conditions_met = 0
            total_conditions = 0
            
            # Volume condition
            if 'volume_ratio_20' in df.columns:
                total_conditions += 1
                volume_ratio = df.loc[i, 'volume_ratio_20']
                if pd.notna(volume_ratio) and pd.api.types.is_numeric_dtype(df['volume_ratio_20'].dtype):
                    try:
                        # Ensure we have a real number, not complex
                        if isinstance(volume_ratio, (int, float)) and not isinstance(volume_ratio, complex):
                            volume_float = float(volume_ratio)
                            if volume_float > 1.5:  # Above average volume
                                conditions_met += 1
                    except (ValueError, TypeError, OverflowError):
                        pass
            
            # RSI condition (not extreme)
            if 'rsi_14' in df.columns:
                total_conditions += 1
                rsi_val = df.loc[i, 'rsi_14']
                if pd.notna(rsi_val) and pd.api.types.is_numeric_dtype(df['rsi_14'].dtype):
                    try:
                        # Ensure we have a real number, not complex
                        if isinstance(rsi_val, (int, float)) and not isinstance(rsi_val, complex):
                            rsi_float = float(rsi_val)
                            if 30 < rsi_float < 70:  # Not oversold or overbought
                                conditions_met += 1
                    except (ValueError, TypeError, OverflowError):
                        pass
            
            # Trend condition (price above short-term MA)
            if 'sma_20' in df.columns and 'close' in df.columns:
                total_conditions += 1
                close_val = df.loc[i, 'close']
                sma_val = df.loc[i, 'sma_20']
                if (pd.notna(close_val) and pd.notna(sma_val) and 
                    pd.api.types.is_numeric_dtype(df['close'].dtype) and
                    pd.api.types.is_numeric_dtype(df['sma_20'].dtype)):
                    try:
                        if (isinstance(close_val, (int, float)) and not isinstance(close_val, complex) and
                            isinstance(sma_val, (int, float)) and not isinstance(sma_val, complex)):
                            close_float = float(close_val)
                            sma_float = float(sma_val)
                            if close_float > sma_float:  # Price above MA
                                conditions_met += 1
                    except (ValueError, TypeError, OverflowError):
                        pass
            # Volatility condition (moderate volatility)
            if 'atr_pct' in df.columns:
                total_conditions += 1
                atr_val = df.loc[i, 'atr_pct']
                if pd.notna(atr_val) and pd.api.types.is_numeric_dtype(df['atr_pct'].dtype):
                    try:
                        # Ensure we have a real number, not complex
                        if isinstance(atr_val, (int, float)) and not isinstance(atr_val, complex):
                            atr_float = float(atr_val)
                            if 1.0 < atr_float < 8.0:  # Moderate volatility
                                conditions_met += 1
                    except (ValueError, TypeError, OverflowError):
                        pass
            
            # Return True if majority of conditions are met
            if total_conditions == 0:
                # Fallback: simple momentum check
                if 'return_5' in df.columns:
                    return_val = df.loc[i, 'return_5']
                    if pd.notna(return_val) and pd.api.types.is_numeric_dtype(df['return_5'].dtype):
                        try:
                            # Ensure we have a real number, not complex
                            if isinstance(return_val, (int, float)) and not isinstance(return_val, complex):
                                return float(return_val) > 0.02  # 2% momentum
                            else:
                                return False
                        except (ValueError, TypeError, OverflowError):
                            return False
                    else:
                        return False
                else:
                    return False
            else:
                # Return True if majority of conditions are met
                return conditions_met / total_conditions >= 0.6  # 60% threshold
                    
        except Exception as e:
            logger.warning(f"Error detecting breakout setup: {e}")
            return False
    
    def label_breakout_targets(self, df: pd.DataFrame, timeframe: str) -> pd.DataFrame:
        """Label breakout targets with realistic profit/loss thresholds"""
        logger.info(f"Labeling breakout targets for {timeframe}")
        
        try:
            # Get timeframe settings
            settings = TIMEFRAME_SETTINGS.get(timeframe, TIMEFRAME_SETTINGS['1d'])
            lookforward_periods = settings['lookforward_periods']
            grace_periods = settings['grace_periods']
            
            # Calculate dynamic targets
            profit_target_pct, stop_loss_pct = self.calculate_dynamic_targets(df, timeframe)
            
            logger.info(f"Using targets - Profit: {profit_target_pct:.1f}%, Stop: {stop_loss_pct:.1f}%")
            
            # Initialize target columns
            df['target'] = 0  # 0 = no breakout, 1 = bullish breakout
            df['target_result'] = 'unknown'  # Will be filled with actual results
            df['target_profit_pct'] = profit_target_pct
            df['target_stop_pct'] = stop_loss_pct
            df['target_hit_period'] = np.nan
            df['target_exit_price'] = np.nan
            
            # Process each row
            total_rows = len(df)
            bullish_breakouts = 0
            profit_hits = 0
            stop_hits = 0
            ties = 0
            
            for i in range(total_rows - lookforward_periods - grace_periods):
                try:
                    current_price = df.loc[i, 'close']
                    
                    # Check if current_price is valid numeric value
                    if (pd.isna(current_price) or 
                        not pd.api.types.is_numeric_dtype(type(current_price)) or
                        not isinstance(current_price, (int, float)) or
                        isinstance(current_price, complex) or
                        current_price <= 0):
                        continue
                    
                    # Convert to float for safety
                    current_price = float(current_price)
                    
                    # Calculate target prices
                    profit_price = current_price * (1 + profit_target_pct / 100)
                    stop_price = current_price * (1 - stop_loss_pct / 100)
                    
                    # Look forward to see if targets are hit
                    end_period = min(i + lookforward_periods + grace_periods, total_rows)
                    future_prices = df.loc[i+1:end_period, ['high', 'low', 'close']].copy()
                    
                    if future_prices.empty:
                        continue
                    
                    # Check for profit target hit (using high prices)
                    profit_hit_mask = future_prices['high'] >= profit_price
                    profit_hit_periods = future_prices[profit_hit_mask].index
                    
                    # Check for stop loss hit (using low prices)
                    stop_hit_mask = future_prices['low'] <= stop_price
                    stop_hit_periods = future_prices[stop_hit_mask].index
                    
                    # Determine what happened first
                    target_result = 'tie'
                    hit_period = np.nan
                    exit_price = future_prices['close'].iloc[-1]  # Default to last price
                    
                    if len(profit_hit_periods) > 0 and len(stop_hit_periods) > 0:
                        # Both hit - which came first?
                        first_profit = profit_hit_periods[0]
                        first_stop = stop_hit_periods[0]
                        
                        if first_profit <= first_stop:
                            target_result = 'profit'
                            hit_period = first_profit - i
                            exit_price = profit_price
                        else:
                            target_result = 'stop'
                            hit_period = first_stop - i
                            exit_price = stop_price
                    elif len(profit_hit_periods) > 0:
                        # Only profit hit
                        target_result = 'profit'
                        hit_period = profit_hit_periods[0] - i
                        exit_price = profit_price
                    elif len(stop_hit_periods) > 0:
                        # Only stop hit
                        target_result = 'stop'
                        hit_period = stop_hit_periods[0] - i
                        exit_price = stop_price
                    else:
                        # Neither hit within timeframe
                        target_result = 'tie'
                    
                    # Determine if this should be labeled as a breakout opportunity
                    is_breakout = self.detect_breakout_setup(df, i, timeframe)
                    
                    if is_breakout and target_result == 'profit':
                        df.loc[i, 'target'] = 1
                        bullish_breakouts += 1
                        profit_hits += 1
                    elif target_result == 'stop':
                        stop_hits += 1
                    else:
                        ties += 1
                    
                    # Store detailed results
                    df.loc[i, 'target_result'] = target_result
                    df.loc[i, 'target_hit_period'] = hit_period
                    df.loc[i, 'target_exit_price'] = exit_price
                    
                except Exception as e:
                    logger.warning(f"Error processing row {i}: {e}")
                    continue
            
            # Calculate statistics
            total_processed = total_rows - lookforward_periods - grace_periods
            breakout_rate = (bullish_breakouts / total_processed * 100) if total_processed > 0 else 0
            profit_rate = (profit_hits / total_processed * 100) if total_processed > 0 else 0
            
            logger.info(f"Labeling complete:")
            logger.info(f"  Total processed: {total_processed}")
            logger.info(f"  Bullish breakouts: {bullish_breakouts} ({breakout_rate:.1f}%)")
            logger.info(f"  Profit hits: {profit_hits} ({profit_rate:.1f}%)")
            logger.info(f"  Stop hits: {stop_hits}")
            logger.info(f"  Ties: {ties}")
            
            return df
            
        except Exception as e:
            logger.error(f"Error in label_breakout_targets: {e}")
            return df
    
    def create_balanced_dataset(self, df: pd.DataFrame) -> pd.DataFrame:
        """Create a balanced dataset for training"""
        try:
            # Count positive and negative samples
            positive_samples = df[df['target'] == 1]
            negative_samples = df[df['target'] == 0]
            
            logger.info(f"Original dataset: {len(positive_samples)} positive, {len(negative_samples)} negative")
            
            # If we have too few positive samples, don't balance
            if len(positive_samples) < 10:
                logger.warning("Too few positive samples for balancing")
                return df
            
            # Balance by undersampling majority class
            min_samples = min(len(positive_samples), len(negative_samples))
            balanced_samples = min_samples * 2  # Keep reasonable size
            
            if balanced_samples > 1000:
                # If too many samples, limit to reasonable size
                samples_per_class = 500
                positive_balanced = positive_samples.sample(n=min(samples_per_class, len(positive_samples)), random_state=42)
                negative_balanced = negative_samples.sample(n=min(samples_per_class, len(negative_samples)), random_state=42)
            else:
                # Use all positive samples and sample negative to match
                positive_balanced = positive_samples
                negative_balanced = negative_samples.sample(n=len(positive_samples), random_state=42)
            
            # Combine and shuffle
            balanced_df = pd.concat([positive_balanced, negative_balanced])
            balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
            
            logger.info(f"Balanced dataset: {len(positive_balanced)} positive, {len(negative_balanced)} negative")
            return balanced_df
            
        except Exception as e:
            logger.error(f"Error creating balanced dataset: {e}")
            return df
    
    def process_symbol_timeframe(self, symbol: str, timeframe: str) -> Optional[pd.DataFrame]:
        """Process a single symbol-timeframe combination"""
        logger.info(f"\n{'='*50}")
        logger.info(f"Processing {symbol} {timeframe}")
        
        try:
            # Load features
            features_df = self.load_features(symbol, timeframe)
            if features_df is None or len(features_df) < 100:
                logger.warning(f"Insufficient data for {symbol} {timeframe}")
                return None
            
            # Label targets
            labeled_df = self.label_breakout_targets(features_df, timeframe)
            
            # Create balanced dataset
            balanced_df = self.create_balanced_dataset(labeled_df)
            
            # Store statistics
            pos_samples = len(balanced_df[balanced_df['target'] == 1])
            neg_samples = len(balanced_df[balanced_df['target'] == 0])
            
            self.labeling_stats[f"{symbol}_{timeframe}"] = {
                'total_samples': len(balanced_df),
                'positive_samples': pos_samples,
                'negative_samples': neg_samples,
                'balance_ratio': pos_samples / neg_samples if neg_samples > 0 else 0,
                'num_features': len([col for col in balanced_df.columns if not col.startswith('target')])
            }
            
            logger.info(f"✅ Processed {symbol} {timeframe}: {len(balanced_df)} samples")
            return balanced_df
            
        except Exception as e:
            logger.error(f"Error processing {symbol} {timeframe}: {e}")
            return None
    
    def save_labeled_data(self, df: pd.DataFrame, symbol: str, timeframe: str) -> Optional[str]:
        """Save labeled data to CSV file"""
        try:
            filename = f"labeled_{symbol}_{timeframe}.csv"
            filepath = os.path.join(LABELED_FOLDER, filename)
            
            df.to_csv(filepath, index=False)
            
            logger.info(f"✅ Saved {len(df)} labeled samples to {filepath}")
            return filepath
            
        except Exception as e:
            logger.error(f"Failed to save labeled data for {symbol} {timeframe}: {e}")
            return None

def process_all_symbols_timeframes(symbols: List[str], timeframes: List[str]) -> Dict:
    """Process all symbol-timeframe combinations"""
    labeler = EnhancedTargetLabeler()
    results = {
        'successful': [],
        'failed': [],
        'stats': {}
    }
    
    total_combinations = len(symbols) * len(timeframes)
    current_combination = 0
    
    logger.info(f"🚀 Starting target labeling for {total_combinations} combinations")
    
    for symbol in symbols:
        for timeframe in timeframes:
            current_combination += 1
            progress = (current_combination / total_combinations) * 100
            
            logger.info(f"\nProgress: {progress:.1f}% ({current_combination}/{total_combinations})")
            
            try:
                # Process symbol-timeframe
                labeled_df = labeler.process_symbol_timeframe(symbol, timeframe)
                
                if labeled_df is not None and not labeled_df.empty:
                    # Save labeled data
                    filepath = labeler.save_labeled_data(labeled_df, symbol, timeframe)
                    
                    if filepath:
                        results['successful'].append(f"{symbol}_{timeframe}")
                    else:
                        results['failed'].append(f"{symbol}_{timeframe}")
                else:
                    results['failed'].append(f"{symbol}_{timeframe}")
                    
            except Exception as e:
                logger.error(f"❌ Failed {symbol} {timeframe}: {e}")
                results['failed'].append(f"{symbol}_{timeframe}")
    
    results['stats'] = labeler.labeling_stats
    return results

def create_labeling_summary_report(results: Dict):
    """Create target labeling summary report"""
    logger.info(f"\n📊 TARGET LABELING REPORT")
    logger.info(f"{'='*60}")
    
    total_samples = 0
    total_positive = 0
    total_negative = 0
    
    for key, stats in results['stats'].items():
        samples = stats['total_samples']
        positive = stats['positive_samples']
        negative = stats['negative_samples']
        balance = stats['balance_ratio']
        
        total_samples += samples
        total_positive += positive
        total_negative += negative
        
        logger.info(f"{key}: {samples} samples ({positive}+ / {negative}-) ratio: {balance:.2f}")
    
    logger.info(f"\n📈 SUMMARY:")
    logger.info(f"Total labeled datasets: {len(results['stats'])}")
    logger.info(f"Total samples: {total_samples:,}")
    logger.info(f"Total positive samples: {total_positive:,}")
    logger.info(f"Total negative samples: {total_negative:,}")
    if total_negative > 0:
        logger.info(f"Overall balance ratio: {total_positive / total_negative:.2f}")

def save_labeling_summary(results: Dict):
    """Save target labeling summary"""
    summary_report = {
        'labeling_timestamp': datetime.now(timezone.utc).isoformat(),
        'timeframe_settings': TIMEFRAME_SETTINGS,
        'successful_labelings': len(results['successful']),
        'failed_labelings': len(results['failed']),
        'successful_files': results['successful'],
        'failed_files': results['failed'],
        'detailed_stats': results['stats']
    }
    
    summary_path = os.path.join(LABELED_FOLDER, 'target_labeling_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary_report, f, indent=2, default=str)
    
    logger.info(f"\n📋 Target labeling summary saved to: {summary_path}")
    return summary_path

def main():
    """Main function to run target labeling"""
    logger.info("🤖 AI CRYPTO TRADING BOT - TARGET LABELING")
    logger.info(f"🕐 {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")
    logger.info(f"👤 User: samannazir55")
    logger.info("=" * 60)
    
    try:
        # Show configuration
        logger.info(f"Configuration:")
        logger.info(f"Base symbols: {BASE_SYMBOLS}")
        logger.info(f"Features folder: {FEATURES_FOLDER}")
        logger.info(f"Labeled folder: {LABELED_FOLDER}")
        logger.info(f"Timeframe settings: {TIMEFRAME_SETTINGS}")
        
        # Check which feature files exist
        available_combinations = []
        for symbol in BASE_SYMBOLS:
            for timeframe in TIMEFRAME_SETTINGS.keys():
                filename = f"features_{symbol}_{timeframe}.csv"
                filepath = os.path.join(FEATURES_FOLDER, filename)
                if os.path.exists(filepath):
                    available_combinations.append((symbol, timeframe))
        
        if not available_combinations:
            logger.error("❌ No feature files found! Run 02_feature_extraction_fixed.py first")
            return False
        
        logger.info(f"Found {len(available_combinations)} feature files to process")
        
        # Extract symbols and timeframes from available combinations
        symbols = list(set([combo[0] for combo in available_combinations]))
        timeframes = list(set([combo[1] for combo in available_combinations]))
        
        # Process all combinations
        results = process_all_symbols_timeframes(symbols, timeframes)
        
        # Show results
        logger.info(f"\n{'='*60}")
        logger.info(f"📊 TARGET LABELING COMPLETE")
        logger.info(f"✅ Successful: {len(results['successful'])}")
        logger.info(f"❌ Failed: {len(results['failed'])}")
        
        if results['failed']:
            logger.info(f"\nFailed labelings:")
            for failed in results['failed']:
                logger.info(f"  - {failed}")
        
        # Create reports
        if results['successful']:
            create_labeling_summary_report(results)
            summary_path = save_labeling_summary(results)
            
            logger.info(f"\n✅ Target labeling completed successfully!")
            logger.info(f"📋 Summary saved to: {summary_path}")
            logger.info(f"📁 Labeled files saved to: {LABELED_FOLDER}/")
            logger.info(f"\nNext step: Run 04_improved_training.py")
        else:
            logger.error(f"\n❌ No targets were successfully labeled!")
            return False
        
        return True
        
    except Exception as e:
        logger.error(f"❌ Target labeling failed: {e}")
        return False

if __name__ == "__main__":
    success = main()
    if not success:
        exit(1)

2025-08-29 03:24:49,259 - INFO - 🤖 AI CRYPTO TRADING BOT - TARGET LABELING
2025-08-29 03:24:49,260 - INFO - 🕐 2025-08-28 22:24:49 UTC
2025-08-29 03:24:49,260 - INFO - 👤 User: samannazir55
2025-08-29 03:24:49,261 - INFO - ============================================================
2025-08-29 03:24:49,261 - INFO - Configuration:
2025-08-29 03:24:49,262 - INFO - Base symbols: ['BTC_USDT', 'ETH_USDT', 'SOL_USDT']
2025-08-29 03:24:49,262 - INFO - Features folder: data/features
2025-08-29 03:24:49,263 - INFO - Labeled folder: data/labeled
2025-08-29 03:24:49,263 - INFO - Timeframe settings: {'6h': {'profit_target_pct': 2.0, 'stop_loss_pct': 1.0, 'lookforward_periods': 4, 'grace_periods': 4}, '12h': {'profit_target_pct': 4.0, 'stop_loss_pct': 2.0, 'lookforward_periods': 4, 'grace_periods': 4}, '1d': {'profit_target_pct': 8.0, 'stop_loss_pct': 4.0, 'lookforward_periods': 5, 'grace_periods': 3}, '3d': {'profit_target_pct': 15.0, 'stop_loss_pct': 8.0, 'lookforward_periods': 5, 'grace_periods': 